# The canopy agent, to play with

This is `demo.py` opened up so you can poke at it: the same agent as the R package's
`inst/demo.R`, over made-up forest canopy data, with no chat UI in front of it.

Launch it from `pkg-py/`:

```bash
export CHATLAS_CHAT_PROVIDER_MODEL=bedrock-anthropic/us.anthropic.claude-sonnet-5
export AWS_PROFILE=claude
uv run --with 'anthropic[bedrock]' --with jupyterlab jupyter lab demo.ipynb
```

If the first model call fails on credentials, refresh the SSO token with
`aws sso login --profile claude`.

## The pieces

`demo.py` holds the two data frames, the notes file, and the two measures, so this
notebook imports them rather than keeping a second copy that can drift. The import
finds `demo.py` wherever the kernel started, so it works from the repository root
as well as from `pkg-py`.

In [ ]:
import sys
from pathlib import Path


# Jupyter's working directory depends on where it was launched, so find the
# directory holding demo.py rather than assuming it is the cwd.
def package_root() -> Path:
    here = Path.cwd()
    for parent in (here, *here.parents):
        for candidate in (parent, parent / "pkg-py"):
            if (candidate / "demo.py").is_file():
                return candidate
    raise FileNotFoundError(f"No demo.py found at or above {here}")


root = package_root()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import demo

print(f"demo.py from {root}")
demo.stands

In [ ]:
demo.surveys.head()

A measure is plain Python that commons trusts. `warehouse` is annotated
`Injected`, so commons supplies the DuckDB connection and the model never sees
that argument. Called directly, a measure is just a function:

In [ ]:
import duckdb

con = duckdb.connect()
con.register('stands', demo.stands)
con.register('surveys', demo.surveys)

demo.canopy_by_county(con)

## The agent

One `Commons` over all three layers: the data source, the semantic layer of two
measures, and the context layer holding the notes prose.

In [ ]:
canopy = demo.agent()
canopy

Which tools registered depends on what you gave it. There is no catalog and no
governed definitions here, so `search_catalog` and `call_metrics` are absent.

`Commons` composes a chatlas `Chat` rather than inheriting from it, so its surface is
curated: `get_tools()` and `system_prompt` are forwarded, and `canopy.client` is the
escape hatch to the rest of chatlas. Ask questions through the agent, though, not
through `client`, which skips the citation scanner and the provenance tag.

In [ ]:
sorted(tool.name for tool in canopy.get_tools())

The system prompt is assembled from the shared `prompts/` source plus everything
commons derived from your sources. Worth reading once.

In [ ]:
print(canopy.system_prompt)

The citation corpus is the trusted text an answer is allowed to quote. An answer
that quotes something outside it does not earn a citation.

In [ ]:
for entry in canopy.citation_corpus():
    print(entry)

## Ask it something

`stream_async` yields the answer as it arrives, then the provenance marker. The
marker is HTML for a UI to render, so here it prints as raw tags.

A question the measures cover should come back `Verified answer`.

In [ ]:
async def ask(agent, question):
    parts = []
    async for chunk in await agent.stream_async(question):
        print(chunk, end='')
        parts.append(chunk)
    return ''.join(parts)


answer = await ask(canopy, 'Which county has the most canopy cover?')

A question no measure covers has to reach `run_sql` instead, so the same agent
answers it `Untrusted`, or `Cited` if it quotes the notes. The `Cited` marker
renders nothing of its own, because the citation aside already says as much.

In [ ]:
answer = await ask(canopy, 'How much canopy has Winberry Ridge gained since 2021?')

Which tools that took:

In [ ]:
demo.tools_run(canopy)

## Your turn

Ask it anything. The notes say canopy is always acre-weighted and that baseline
statistics cover established stands only, so questions that brush against those
rules are the interesting ones.

`canopy` keeps the conversation, so follow-up questions have the earlier turns.

In [ ]:
answer = await ask(canopy, 'Is a plain average of canopy_pct across stands wrong? Why?')

To try a measure of your own, add it to the semantic layer and rebuild. What that
changes is not the tool list, since `call_measure` is already registered, but what
`search_pool` can find and therefore what earns a verified answer.

In [ ]:
from typing import Any

import commons


@commons.measure(description='Total acres by forest type, largest first.')
def acres_by_forest_type(warehouse: commons.Injected[Any]):
    return warehouse.execute(
        'SELECT forest_type, SUM(acres) AS acres FROM stands'
        ' GROUP BY forest_type ORDER BY acres DESC'
    ).fetchdf()


layer = commons.semantic_layer(
    demo.canopy_by_county, demo.low_canopy_stands, acres_by_forest_type
)

mine = commons.Commons(
    demo.client(),
    {'warehouse': commons.data_source(stands=demo.stands, surveys=demo.surveys)},
    semantic_layer=layer,
    context_layer=commons.context_layer(files=[demo.notes_file()]),
)

sorted(layer.measures)

The question below has no measure behind it in `canopy`, so that agent would have to
reach `run_sql`. `mine` has one, so watch which tools it runs and which marker it
ends with.

In [ ]:
answer = await ask(mine, 'How many acres are ponderosa pine?')